# From local book and trade files to features

Run these cells only after adding competition files to `data/raw` and installing the package from the README. They read one stock for explanation and do not train a model. The notebook is delivered unexecuted.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the repository or notebooks directory"


In [ ]:
from optiver_volatility.config import load_config
from optiver_volatility.data import read_index, read_partition
from optiver_volatility.features import book_features, stock_features
cfg = load_config(ROOT / "configs/default.json")
index = read_index(ROOT / "data/raw", "train")
index.head()


## Read one stock partition

The stock key is encoded in the partition directory. Load only the required columns, then sort by window and elapsed second. Do not sort all stocks together and calculate one global difference.


In [ ]:
stock = int(index.stock_id.min())
time_ids = index.loc[index.stock_id == stock, "time_id"].tolist()
book = read_partition(ROOT / "data/raw", "book", "train", stock, time_ids)
trade = read_partition(ROOT / "data/raw", "trade", "train", stock, time_ids)
print("Book records:", len(book), "Trade records:", len(trade))
book.head()


## Inspect the observed window

The figure describes input data only. It does not visualize the hidden target window. WAP uses top-level quotes and opposite-side sizes.


In [ ]:
one = book.loc[book.time_id == time_ids[0]].copy()
one["wap"] = (one.bid_price1 * one.ask_size1 + one.ask_price1 * one.bid_size1) / (one.bid_size1 + one.ask_size1)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(one.seconds_in_bucket, one.wap, color="#185B4D", linewidth=1.2)
ax.set(xlabel="Seconds in observed window", ylabel="WAP", title=f"Observed input: stock {stock}, time {time_ids[0]}")
ax.grid(alpha=.2)
plt.tight_layout()


## Summarize book and trade observations

The full window and the last 300/120 seconds provide different views of recent activity. Returns are recomputed inside each subwindow. Means use reported updates, not a forward-filled second grid.

No trades means zero trade activity and missing trade-price statistics. A missing book window is an error. No future targets enter feature calculations.


In [ ]:
features = stock_features(book, trade, stock, time_ids, cfg["window_starts"])
features.head()


In [ ]:
learning_table = index.loc[index.stock_id == stock].merge(
    features, on=["stock_id", "time_id"], validate="one_to_one")
learning_table[["stock_id", "time_id", "target", "book_w600_wap1_rv",
                "book_w300_wap1_rv", "trade_w600_volume", "trade_w600_has_trade"]].head()


## Build the full feature cache later

From a terminal at the repository root:

```bash
optiver-lab build-features --split train
```

This writes `data/processed/train.parquet` and its manifest. For a small first run, use `--max-stocks 3 --output-dir data/processed/small`; it is a learning subset, not a representative performance sample. Keep the full cache separate.

`time_id`, `row_id` and `target` are excluded from model inputs. `stock_id` is categorical in LightGBM.
